In [2]:
!pip install -q langchain
!pip install -q langgraph
!pip install -q crewai
!pip install -q google-generativeai
!pip install -q pytesseract
!pip install -q pdf2image
!pip install -q pillow
!pip install -q pandas
!pip install -q chromadb
!pip install -q sentence-transformers
!apt-get install -y poppler-utils
!apt-get install -y tesseract-ocr

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.


In [11]:
!pip install -q groq
!pip install -q pytesseract pdf2image pillow pandas
!pip install -q langchain langgraph crewai
!pip install -q chromadb sentence-transformers

!apt-get install -y poppler-utils
!apt-get install -y tesseract-ocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 7.2 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.


In [17]:
from groq import Groq
from google.colab import userdata

# Retrieve API key from Colab secrets
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

client = Groq(
    api_key=GROQ_API_KEY
)

In [18]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Hello"}
    ]
)

print(response.choices[0].message.content)

Hello. How can I assist you today?


In [23]:
from groq import Groq
from google.colab import userdata

# Ensure GROQ_API_KEY is retrieved, assuming it's already in userdata if 2plaetiFxExR ran
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key=GROQ_API_KEY)

def ask_groq(prompt):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [21]:
import pytesseract
from PIL import Image

img = Image.open("page_0.jpeg")

invoice_text = pytesseract.image_to_string(img)

print(invoice_text[:2000])

IN Wie] el Ss

 

 

i To
Invoice No. ent Company Name
Date of lesue Ener Oste Here ess
em ‘Description Hours Rate Amount
fi {ago denn a $05) s8000

Subtotal $1500,
Tax Rate 00%
ter s750
Toul $157.50



In [24]:
prompt = f"""
You are an Invoice Extraction Agent.

Extract:

- Vendor Name
- Invoice Number
- Invoice Date
- Due Date
- Tax Amount
- Total Amount
- Currency

Return ONLY valid JSON.

Invoice Text:
{invoice_text}
"""

result = ask_groq(prompt)

print(result)

```json
{
  "Vendor Name": "Ener Oste Here ess",
  "Invoice Number": "ent",
  "Invoice Date": null,
  "Due Date": null,
  "Tax Amount": 0.00,
  "Total Amount": 157.50,
  "Currency": "USD"
}
```


In [25]:
import json
import re

json_string = re.search(
    r'\{.*\}',
    result,
    re.DOTALL
).group()

invoice_data = json.loads(json_string)

print(invoice_data)

{'Vendor Name': 'Ener Oste Here ess', 'Invoice Number': 'ent', 'Invoice Date': None, 'Due Date': None, 'Tax Amount': 0.0, 'Total Amount': 157.5, 'Currency': 'USD'}


In [26]:
validation_prompt = f"""
You are a Validation Agent.

Check:

1. Missing fields
2. Invalid dates
3. Negative amounts
4. Missing invoice number

Invoice:

{invoice_data}

Provide a validation report.
"""

validation_report = ask_groq(validation_prompt)

print(validation_report)

Validation Report:

1. Missing fields: 
   - Invoice Date is missing (None)
   - Due Date is missing (None)

2. Invalid dates: 
   - Invoice Date is None, which is invalid
   - Due Date is None, which is invalid

3. Negative amounts: 
   - No negative amounts found

4. Missing invoice number: 
   - Invoice Number is present but may be invalid ('ent' might not be a valid invoice number, however, it's not empty)

Additional recommendation: 
- Verify the Invoice Number ('ent') to ensure it matches the expected format and is a valid identifier for the invoice. 

The invoice is invalid due to missing and invalid date fields. These issues must be addressed before the invoice can be considered valid.


In [28]:
fraud_prompt = f"""
You are a Fraud Detection Agent.

Analyze:

- Fake invoice indicators
- Duplicate invoice possibility
- Suspicious amount
- Vendor authenticity concerns

Invoice:

{invoice_data}

Give:

Risk Score: 0-100

Reasoning
"""

fraud_report = ask_groq(fraud_prompt)

print(fraud_report)

**Risk Score: 95**

**Reasoning:**

The provided invoice raises several red flags, indicating a high risk of fraud. Here's a breakdown of the concerns:

1. **Fake invoice indicators:**
	* The 'Vendor Name' appears to be a jumbled collection of words, which may indicate a fabricated or non-existent vendor.
	* The 'Invoice Number' is extremely short and lacks a standard format, which is unusual for legitimate invoices.
2. **Duplicate invoice possibility:**
	* Without a unique 'Invoice Number' or 'Invoice Date', it's challenging to verify whether this invoice is a duplicate or not.
3. **Suspicious amount:**
	* The 'Total Amount' of $157.50 is a relatively small amount, which could be an attempt to fly under the radar. However, the lack of context or supporting documentation makes it difficult to determine the legitimacy of this amount.
4. **Vendor authenticity concerns:**
	* The 'Vendor Name' is highly suspicious, and the absence of 'Invoice Date' and 'Due Date' fields raises concerns abo

In [29]:
approval_prompt = f"""
Invoice Details:

{invoice_data}

Validation:

{validation_report}

Fraud Analysis:

{fraud_report}

Should this invoice be:

APPROVED
REJECTED
REVIEW REQUIRED

Explain.
"""

approval_result = ask_groq(approval_prompt)

print(approval_result)

Based on the provided information, this invoice should be **REJECTED**.

The invoice has several red flags that indicate a high risk of fraud, including:

1. A suspicious vendor name that appears to be fabricated.
2. A short and potentially invalid invoice number.
3. Missing critical fields, such as the invoice date and due date.
4. A relatively small total amount that could be an attempt to avoid detection.

The validation report also highlights several issues, including missing and invalid date fields, which render the invoice invalid.

The fraud analysis assigns a high risk score of 95, indicating a strong likelihood of fraud. The recommendation is to verify the vendor's existence and authenticity, as well as request supporting documentation to validate the invoice's legitimacy. However, given the numerous red flags and the high risk score, it is more prudent to reject the invoice outright rather than investing time and resources into reviewing and verifying its legitimacy.

Therefo

In [30]:
import pandas as pd

df = pd.DataFrame([invoice_data])

df["validation"] = validation_report
df["fraud_report"] = fraud_report
df["approval"] = approval_result

df.to_csv("processed_invoice.csv", index=False)

df

,Vendor Name,Invoice Number,Invoice Date,Due Date,Tax Amount,Total Amount,Currency,validation,fraud_report,approval
0,Ener Oste Here ess,ent,None,None,0.0,157.5,USD,Validation Report:\n\n1. Missing fields: \n ...,**Risk Score: 95**\n\n**Reasoning:**\n\nThe pr...,"Based on the provided information, this invoic..."
